<a href="https://colab.research.google.com/github/farrelrassya/Build-a-Large-Language-Model-From-Scratch/blob/main/03.Coding_Attention_Mechanisms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farrelrassya/Build-a-Large-Language-Model-From-Scratch/blob/main/03.Coding_Attention_Mechanisms.ipynb)

# Chapter 3 -- Coding Attention Mechanisms

*Build a Large Language Model (From Scratch)* -- Sebastian Raschka (Manning, 2024)

**Technical edition -- setiap output di notebook ini dieksekusi riil di mesin builder; tidak ada output yang difabrikasi.**

Bab 3 membangun komponen inti GPT: attention. Ada empat tahap. Pertama self-attention sederhana tanpa bobot latih, lalu scaled dot-product attention dengan matriks $W_q$, $W_k$, $W_v$. Setelah itu causal mask plus dropout, dan terakhir multi-head attention. Semua kode bab adalah listing dari companion repo `rasbt/LLMs-from-scratch` (commit `cdbd33e`, Apache-2.0), diambil verbatim dan disusun sesuai urutan buku. Narasi di sini parafrase, bukan salinan prosa buku.

**Catatan kejujuran.** Notebook ini dibangun di Apple M1 Pro, Darwin 25.5.0 arm64, Python 3.14.6, PyTorch 2.14.1. Semua tensor bab dibuat di CPU, sesuai default PyTorch, sehingga semua angka bab adalah angka CPU. Variabel `device` dipilih otomatis (`cuda > mps > cpu`, di mesin build = `mps`). Satu-satunya pemakainya adalah satu sel UTH 3 yang membandingkan RNG dropout antar-backend. Rinciannya sebagai berikut.

- **Book-match:** 633 nilai numerik dari 34 output cetak dibandingkan digit-per-digit dengan output live. Hasilnya semuanya cocok.
- **Repo-match:** 43/45 sel repo menghasilkan teks output yang identik dengan output tersimpan di repo (torch 2.4.0). Satu-satunya pengecualian adalah dua sel yang mencetak versi torch.
- **Replay:** tidak ada. Script `replay/run_dropout_linux.sh` mencoba menjalankan fingerprint dropout di container Docker Linux, tetapi macet di credential helper Docker Desktop sebelum image `python:3.14-slim` tertarik. Run dihentikan tanpa output, jadi notebook tidak mengklaim angka Linux apa pun.
- **Kode yang sengaja error:** satu sel. Isinya memverifikasi klaim komentar repo bahwa input yang lebih panjang dari `context_length` akan crash.
- **Unexecuted by design:** tidak ada.

| Under the hood | What we measure | Result |
|---|---|---|
| 1. Kenapa skor dibagi $\sqrt{d_k}$ | std skor $q \cdot k$ vs $d_k$, bobot softmax terbesar, norma Jacobian softmax terhadap logit-nya, dengan dan tanpa skala | std skor mengikuti $\sqrt{d_k}$ (27.5 vs 27.7 di $d_k$=768). Tanpa skala, bobot max naik 0.46 -> 0.986 dan $\|J\|$ (Jacobian softmax terhadap logit-nya) turun 19x (0.368 -> 0.0197). Dengan skala, $\|J\|$ terhadap logit $s/\sqrt{d_k}$ stabil di 0.380-0.386 |
| 2. Causal mask tidak membocorkan masa depan | token 4-6 diganggu $+10\sigma$; Jacobian $\partial z_t / \partial x_s$; dua rute masking buku | token 1-3 bit-identik setelah token 4-6 diganggu (token 4-6 bergeser s.d. 12.59); Jacobian segitiga atas tepat 0.0, tanpa mask 0/36 entri nol |
| 3. Dropout: RNG, backend, train vs eval | mask buku vs uniform float64; CPU vs akselerator; rowsum saat train; `no_grad` vs `eval()` | mask buku = `2*(u64<0.5)` di macOS arm64 tanpa MKL; MPS memberi mask lain (14/36 nol); saat train rowsum bukan 1 (undian 1: 2.00, 0.00, 2.00, 0.98, 0.77, 1.31), rata-rata 10k 1.000; `no_grad` tetap drop, hanya `eval()` yang identitas |
| 4. Wrapper vs `MultiHeadAttention` | transplantasi bobot antar-implementasi; jumlah parameter; latency CPU skala GPT-2; cross-check SDPA | transplantasi bobot: max selisih 6.0e-08; skala GPT-2 (b=2, T=1024, CPU): wrapper 87 ms vs MHA 52 ms vs SDPA 18 ms; buffer mask wrapper 12x lebih besar |

## Setup

Sel pertama adalah sel versi dari repo. Sel kedua (kode orisinal) mencatat mesin, backend yang tersedia, dan device yang dipakai.

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.11.0+cu130


Repo menyimpan output `torch version: 2.4.0`. Notebook ini berjalan di `2.14.1`, sepuluh rilis minor kemudian. Tabel book-match dan repo-match di bawah menunjukkan bahwa selisih versi ini tidak mengubah satu angka pun di bab 3.

In [2]:
import os
import platform
import subprocess
import torch

device = torch.device(os.environ.get("NB_DEVICE") or
                      ("cuda" if torch.cuda.is_available() else
                       "mps" if torch.backends.mps.is_available() else "cpu"))

cpu_name = platform.processor()
if platform.system() == "Darwin":
    cpu_name = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                              capture_output=True, text=True).stdout.strip()
HOST_LABEL = f"{cpu_name}, {platform.system()} {platform.release()} {platform.machine()}"

print(f"host    : {HOST_LABEL}")
print(f"python  : {platform.python_version()} | torch {torch.__version__}")
print(f"backend : cuda={torch.cuda.is_available()} mps={torch.backends.mps.is_available()} "
      f"MKL={torch.backends.mkl.is_available()} threads={torch.get_num_threads()}")
print(f"device  : {device}" + ("  (override NB_DEVICE)" if os.environ.get("NB_DEVICE") else ""))


host    : x86_64, Linux 6.6.122+ x86_64
python  : 3.13.15 | torch 2.11.0+cu130
backend : cuda=True mps=False MKL=True threads=1
device  : cuda


Mesin build adalah Apple Silicon dengan MPS. Listing bab tidak pernah memindahkan tensor ke `device`, jadi semua perhitungan bab berjalan di CPU. `NB_DEVICE=cpu` memaksa jalur CPU murni. Build torch ini juga melaporkan `MKL=False`. Detail ini penting di UTH 3, karena RNG dropout di CPU bergantung pada build library, bukan pada kode bab.

Perbedaan versi antara buku cetak, repo, dan notebook ini:

| Aspek | Buku cetak | Repo `cdbd33e` | Notebook ini |
|---|---|---|---|
| PyTorch | tidak dicetak | output tersimpan 2.4.0 | 2.14.1, Python 3.14 |
| Bentuk listing | class dan pemakaiannya dipisah | class + pemakaian dalam satu sel | ikut repo, verbatim |
| `d_k` di 3.4.1 | `keys.shape[-1]` | `keys.shape[1]` | repo; ekuivalen untuk tensor 2D (diukur di 3.5.1) |
| softmax di 3.5.1 | `dim=1` | `dim=-1` | repo; ekuivalen untuk tensor 2D (diukur) |
| Listing 3.3 | hanya mencetak shape | mencetak tensor + shape | repo |
| Solusi Exercise 3.2-3.3 | -- | blok kode di sel markdown, tidak dieksekusi | dieksekusi live di bagian Exercises |

Bab ini dibuka dengan peta jalan. Attention adalah langkah 2 dari stage 1 (figure 3.1), dibangun dalam empat varian yang saling menumpuk (figure 3.2).

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.1.png" width="700">

**Figure 3.1** Peta tiga stage membangun LLM. Bab ini mengerjakan langkah 2 stage 1: mekanisme attention, komponen di dalam arsitektur LLM.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.2.png" width="700">

**Figure 3.2** Empat varian attention yang dibangun berurutan: simplified self-attention, self-attention dengan bobot latih, causal attention, dan multi-head attention.

## 3.1 The problem with modeling long sequences

Terjemahan kata per kata gagal karena tiap bahasa menyusun kalimat dengan urutan berbeda. Kata kerja Jerman bisa berada di ujung kalimat, padahal di bahasa Inggris ia muncul lebih awal (figure 3.3). Sebelum transformer, jawaban standarnya adalah encoder-decoder RNN. Encoder membaca seluruh kalimat sumber dan memadatkannya ke satu hidden state. Dari state itu decoder menulis terjemahan token demi token (figure 3.4).

Titik lemahnya ada di pemadatan. Saat decoding, decoder tidak bisa melihat hidden state encoder yang lebih awal. Seluruh makna kalimat harus muat di satu vektor terakhir, sehingga dependensi jarak jauh mudah hilang.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.3.png" width="700">

**Figure 3.3** Terjemahan Jerman ke Inggris kata per kata menghasilkan kalimat yang salah tata bahasa. Terjemahan yang benar perlu melihat kata yang muncul lebih awal atau lebih akhir di kalimat sumber.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.4.png" width="700">

**Figure 3.4** Encoder-decoder RNN: encoder memadatkan seluruh input ke satu hidden state, lalu decoder menghasilkan output token demi token dari state tersebut.

## 3.2 Capturing data dependencies with attention mechanisms

Bahdanau attention (2014) mengubah encoder-decoder RNN supaya decoder bisa mengakses semua posisi input di setiap langkah. Setiap input diberi bobot kepentingan untuk token output yang sedang dibuat (figure 3.5). Tiga tahun kemudian, arsitektur transformer membuang RNN sepenuhnya dan mempertahankan ide ini sebagai *self-attention*. Di sana setiap posisi dalam satu sekuens menimbang semua posisi lain di sekuens yang sama. Figure 3.6 menunjukkan letak modul ini di dalam GPT decoder-only. Bagian lain arsitekturnya dibangun di bab 4.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.5.png" width="700">

**Figure 3.5** Dengan attention, decoder mengakses semua token input secara selektif. Tebal garis mewakili seberapa penting sebuah token input untuk token output yang sedang dihasilkan.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.6.png" width="700">

**Figure 3.6** Self-attention adalah modul di dalam GPT decoder-only. Preprocessing (bab 2) ada di sebelum modul ini dan sisa arsitektur (bab 4) di sekitarnya.

## 3.3 Attending to different parts of the input with self-attention

### 3.3.1 A simple self-attention mechanism without trainable weights

Kata "self" berarti bobot attention dihitung antar-posisi di dalam *satu* sekuens. Ini berbeda dengan attention seq2seq yang menghubungkan dua sekuens. Targetnya adalah sebuah *context vector* $z^{(i)}$ untuk setiap embedding input $x^{(i)}$. Context vector ini adalah rata-rata berbobot dari semua input, jadi embedding token yang sudah diperkaya informasi token lain. Contoh berjalan bab ini adalah kalimat "Your journey starts with one step": enam token dengan embedding 3 dimensi. Varian pertama ini belum punya bobot latih. Fungsinya untuk memperkenalkan tiga langkah dasar: skor, bobot, context vector.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.7.png" width="700">

**Figure 3.7** Context vector $z^{(2)}$ adalah kombinasi semua input yang ditimbang relatif terhadap $x^{(2)}$. Bobot $\alpha_{21} \ldots \alpha_{2T}$ menentukan kontribusi masing-masing input.

In [3]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

Satu baris per token, satu kolom per dimensi embedding. Langkah 1 menghitung *attention score* $\omega_{2j} = x^{(2)} \cdot x^{(j)}$ dengan token kedua ("journey") sebagai query.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.8.png" width="700">

**Figure 3.8** Langkah 1: skor attention adalah dot product antara query $x^{(2)}$ dan setiap input. Angka di gambar dipotong ke satu desimal.

In [4]:
query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Skor untuk query "journey" adalah `[0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]`, cocok digit-per-digit dengan buku (hlm. 58). Skor tertinggi 1.4950 adalah dot product journey dengan dirinya sendiri, yaitu kuadrat panjangnya: $0.55^2 + 0.87^2 + 0.66^2 = 0.3025 + 0.7569 + 0.4356 = 1.4950$. Peringkat kedua adalah "starts" (1.4754), karena embedding-nya `[0.57, 0.85, 0.64]` hampir sama dengan journey. Dot product mengukur arah dan panjang sekaligus, tanpa normalisasi seperti cosine similarity. Konsekuensinya terlihat di 3.3.2.

In [5]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


Loop manual dan `torch.dot` sama-sama memberi 0.9544: $0.43 \cdot 0.55 + 0.15 \cdot 0.87 + 0.89 \cdot 0.66 = 0.2365 + 0.1305 + 0.5874$. Dot product tidak lebih dari kali-elemen lalu jumlah.

Langkah 2 menormalisasi skor menjadi *attention weight* yang berjumlah 1.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.9.png" width="700">

**Figure 3.9** Langkah 2: skor $\omega$ dinormalisasi menjadi bobot attention $\alpha$ yang berjumlah 1.

In [6]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


Normalisasi paling sederhana membagi setiap skor dengan totalnya, $6.5617$. Contohnya $0.9544 / 6.5617 = 0.1455$. Rasio antarbobot di sini sama dengan rasio antarskor: $0.2278 / 0.1077 = 2.115 = 1.4950 / 0.7070$. Kelemahannya, skor negatif akan menghasilkan bobot negatif dan nilai ekstrem tidak diredam. Perhatikan juga format jumlahnya, `tensor(1.0000)` dan bukan `tensor(1.)`. Selisih kecil ini diukur dua sel lagi.

In [7]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


Softmax memberi `[0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]`, sama dengan buku. Rasio bobot sekarang mengikuti *selisih* skor, bukan rasionya: $e^{1.4950 - 0.7070} = e^{0.788} = 2.199 = 0.2379 / 0.1082$. Dibanding normalisasi biasa, token dengan skor tertinggi naik dari 0.2278 ke 0.2379. Semua bobot dijamin positif.

In [8]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


`torch.softmax` menghasilkan angka yang sama persis dengan `softmax_naive` untuk skor sekecil ini. Buku menyebut versi naif rawan overflow pada input besar. Sel berikut (kode orisinal) mengukur klaim itu, sekaligus selisih jumlah `1.0000` vs `1.`.

In [9]:
x_big = torch.tensor([1000.0, 1001.0, 999.0])
print("softmax_naive(x_big)       :", softmax_naive(x_big))
print("torch.softmax(x_big)       :", torch.softmax(x_big, dim=0))
print("softmax_naive(x_big - max) :", softmax_naive(x_big - x_big.max()))
print("float32 exp overflow mulai :", torch.finfo(torch.float32).max, "-> ln =", round(torch.log(torch.tensor(torch.finfo(torch.float32).max)).item(), 2))
print("sum normalisasi biasa      :", attn_weights_2_tmp.sum().item())
print("sum torch.softmax          :", attn_weights_2.sum().item())


softmax_naive(x_big)       : tensor([nan, nan, nan])
torch.softmax(x_big)       : tensor([0.2447, 0.6652, 0.0900])
softmax_naive(x_big - max) : tensor([0.2447, 0.6652, 0.0900])
float32 exp overflow mulai : 3.4028234663852886e+38 -> ln = 88.72
sum normalisasi biasa      : 1.0000001192092896
sum torch.softmax          : 1.0


Versi naif menghasilkan `nan` di ketiga posisi. Penyebabnya, $e^{1000}$ melampaui batas float32: `exp` sudah overflow di atas $\ln(3.4 \times 10^{38}) \approx 88.72$, dan $\infty / \infty$ = `nan`. `torch.softmax` memberi `[0.2447, 0.6652, 0.0900]`. Hasil yang sama persis muncul dari versi naif setelah input dikurangi maksimumnya. Itulah trik stabilisasi standar, dan ia sah karena softmax invarian terhadap pergeseran konstan: $\text{softmax}(x - c) = \text{softmax}(x)$.

Baris terakhir menjelaskan format cetakan tadi. Jumlah normalisasi biasa adalah 1.0000001192092896, tepat $1 + 2^{-23}$, satu ulp float32 di atas 1, sehingga PyTorch mencetak `1.0000`. Jumlah softmax tepat 1.0, sehingga dicetak `1.`.

Langkah 3 menjumlahkan semua input yang sudah dikalikan bobotnya.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.10.png" width="700">

**Figure 3.10** Langkah 3: context vector $z^{(2)}$ adalah jumlah semua vektor input yang masing-masing dikalikan bobot attention-nya.

In [10]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


$z^{(2)} = $ `[0.4419, 0.6515, 0.5683]`, cocok dengan buku (hlm. 60). Bobotnya positif dan berjumlah 1, jadi setiap komponen $z^{(2)}$ adalah rata-rata berbobot kolom yang bersesuaian. Nilainya pasti berada di antara minimum dan maksimum kolom itu. Contohnya kolom pertama input berkisar 0.05-0.77, dan 0.4419 ada di dalamnya. Journey dan starts menyumbang bobot terbesar, $0.2379 + 0.2333 = 0.4712$, sehingga $z^{(2)}$ paling mirip dengan kedua embedding itu.

### 3.3.2 Computing attention weights for all input tokens

Ketiga langkah tadi sekarang diulang untuk semua token sekaligus. Hasilnya matriks bobot $6 \times 6$, dan baris kedua adalah hasil yang baru saja dihitung (figure 3.11, figure 3.12).

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.11.png" width="700">

**Figure 3.11** Matriks bobot attention untuk semua pasangan token. Baris yang disorot adalah bobot untuk query "journey". Angka dipotong ke dua desimal.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.12.png" width="700">

**Figure 3.12** Tiga langkah yang sama -- skor, bobot, context vector -- kini untuk semua input. Langkah 1 mendapat loop tambahan.

In [11]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Matriks skor $\Omega = XX^\top$ simetris ($\omega_{ij} = \omega_{ji}$), dan diagonalnya adalah kuadrat panjang setiap embedding: 0.9995, 1.4950, 1.4570, 0.4937, 0.6654, 0.9450. Baris kedua sama dengan `attn_scores_2`.

Ada temuan yang terbaca langsung dari angka ini: token *tidak* selalu memberi skor tertinggi pada dirinya sendiri. Hanya "Your" dan "journey" yang begitu. "starts" memberi journey 1.4754, lebih tinggi dari dirinya sendiri (1.4570). "one" memberi starts 0.7154 di atas dirinya (0.6654). "with" dan "step" sama-sama paling tinggi ke journey (0.8434 dan 1.0865). Dot product mentah menghadiahi vektor yang panjang. Ini salah satu alasan bobot latih $W_q$ dan $W_k$ diperkenalkan di 3.4: model perlu bebas mempelajari kemiripan yang berguna.

In [12]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Perkalian matriks `inputs @ inputs.T` memberi matriks yang sama dengan nested loop (buku hlm. 62-63, digit-per-digit). Loop Python $O(T^2)$ diganti satu kernel BLAS. Pada $T = 1024$ (konteks GPT-2), loop itu berarti 1,048,576 iterasi interpreter per kalimat per head.

In [13]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


`dim=-1` menormalisasi setiap baris. Baris kedua `[0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]` identik dengan `attn_weights_2` dari 3.3.1. Bobot terbesar per baris mengikuti pola skor: baris "starts" paling besar ke journey (0.2369), bukan ke dirinya (0.2326).

In [14]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


Baris pertama menjumlahkan enam angka cetakan yang sudah dibulatkan empat desimal, dan hasilnya tepat 1.0 sebagai float Python. Baris kedua menjumlahkan tensor penuh. Keenam baris bernilai `1.0000`, cocok dengan buku.

In [15]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


Setiap baris adalah context vector satu token, dan baris kedua `[0.4419, 0.6515, 0.5683]` adalah $z^{(2)}$. Context vector jauh lebih seragam daripada input: kolom pertama input berkisar 0.05-0.77 (lebar 0.72), sedangkan kolom pertama context berkisar 0.4177-0.4671 (lebar 0.049), sekitar 15 kali lebih sempit. Tanpa bobot latih, attention di sini hanya meratakan. Bobot latih di 3.4 memberi model kendali atas *apa* yang dirata-ratakan dan dengan bobot berapa.

In [16]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


Hasil loop di 3.3.1 sama persis dengan baris kedua hasil matriks. Implementasi vektor dan implementasi loop konsisten.

## 3.4 Implementing self-attention with trainable weights

Varian berikutnya adalah *scaled dot-product attention*, mekanisme yang dipakai transformer asli dan GPT. Bedanya dengan 3.3 ada pada tiga matriks bobot latih: $W_q$, $W_k$, $W_v$. Setiap embedding diproyeksikan menjadi query, key, dan value. Skor dihitung antara query dan key, lalu context vector adalah rata-rata berbobot dari *value*, bukan dari input mentah. Bobot ini diperbarui saat training (bab 5), sehingga model belajar sendiri ukuran kemiripan dan isi yang diambil.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.13.png" width="700">

**Figure 3.13** Posisi bagian ini di peta bab: simplified self-attention selesai, sekarang ditambah bobot latih. Causal mask dan multi-head menyusul.

### 3.4.1 Computing the attention weights step by step

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.14.png" width="700">

**Figure 3.14** Langkah pertama dengan bobot latih: input $x^{(i)}$ dikalikan $W_q$, $W_k$, $W_v$ menjadi query, key, value. Token kedua tetap menjadi query.

In [17]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

`d_in = 3` adalah lebar embedding input dan `d_out = 2` adalah lebar proyeksi. Di GPT keduanya sama ($d_{in} = d_{out} = 768$ untuk GPT-2 small). Di sini sengaja dibedakan supaya perubahan shape mudah diikuti.

In [18]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

Tiga matriks $3 \times 2$ diambil dari `torch.rand` (uniform $[0, 1)$) dengan seed 123. `requires_grad=False` hanya membuat output lebih bersih; untuk training flag itu harus `True`. Istilah "weight" punya dua arti di bab ini. *Weight parameter* adalah koefisien jaringan yang dilatih ($W_q, W_k, W_v$). *Attention weight* adalah bobot dinamis per input yang dihitung ulang di setiap forward pass.

In [19]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


$q^{(2)} = x^{(2)} W_q = $ `[0.4306, 1.4551]`, cocok dengan buku (hlm. 66). Embedding 3 dimensi diproyeksikan ke 2 dimensi karena $W_q$ punya dua kolom.

In [20]:
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


Walaupun target sementara hanya $z^{(2)}$, key dan value untuk *semua* token tetap dibutuhkan, karena skor query 2 dihitung terhadap setiap key. Keenam token diproyeksikan dalam satu perkalian matriks: `(6, 3) @ (3, 2) -> (6, 2)`.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.15.png" width="700">

**Figure 3.15** Langkah skor: dot product antara query $q^{(2)}$ dan setiap key $k^{(j)}$, bukan lagi antar-embedding mentah.

In [21]:
keys_2 = keys[1] # Python starts index at 0
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


Skor tak-ternormalisasi $\omega_{22} = q^{(2)} \cdot k^{(2)} = 1.8524$ (buku hlm. 68). Itu satu skor; sel berikut menghitung keenamnya dengan satu perkalian matriks.

In [22]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


Baris lengkap untuk query 2 adalah `[1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440]`. Elemen keduanya sama dengan $\omega_{22}$. Kedua sel cocok digit-per-digit dengan buku (hlm. 68). Polanya mirip 3.3 karena $W_q$ dan $W_k$ sama-sama berisi angka positif dari `torch.rand`. Journey dan starts masih unggul, tapi selisihnya sekarang ditentukan proyeksi, bukan embedding mentah.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.16.png" width="700">

**Figure 3.16** Skor attention dinormalisasi dengan softmax menjadi bobot attention. Sebelum softmax, skor dibagi $\sqrt{d_k}$.

In [23]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Bobot untuk query 2 adalah `[0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820]`, cocok dengan buku (hlm. 69). Skor dibagi $\sqrt{d_k} = \sqrt{2} = 1.4142$ sebelum softmax. Repo memakai `keys.shape[1]`, buku memakai `keys.shape[-1]`. Untuk `keys` 2D berukuran `(6, 2)` keduanya bernilai 2.

Buku memberi alasan pembagian ini dalam satu paragraf. Kalau dimensi besar, dot product membesar, softmax berubah menjadi hampir fungsi tangga, dan gradien mendekati nol. UTH 1 mengukur ketiga mata rantai klaim itu.

### Under the hood 1 -- kenapa skor dibagi $\sqrt{d_k}$

Asumsinya diambil dari paper transformer: komponen $q$ dan $k$ saling bebas dengan mean 0 dan varians 1. Untuk setiap $d_k$, sel ini mengambil 1000 query acak dan 6 key (sama dengan jumlah token contoh bab). Lalu ia mengukur tiga hal. Pertama std skor $q \cdot k$. Kedua rata-rata bobot softmax terbesar. Ketiga norma Frobenius $J = \partial p / \partial \ell = \text{diag}(p) - pp^\top$, yaitu Jacobian softmax terhadap logit $\ell$ yang *masuk* softmax: $\ell = s$ tanpa skala, $\ell = s / \sqrt{d_k}$ dengan skala. Metrik ini mengukur saturasi softmax. Turunan terhadap skor mentah $s$ adalah $J$ untuk versi tanpa skala, tetapi $J / \sqrt{d_k}$ untuk versi berskala. Generator lokal dipakai supaya RNG global bab tidak tersentuh.

In [24]:
uth_gen = torch.Generator().manual_seed(123)
T_tok, N_q = 6, 1000
uth1 = []
print(f"{'d_k':>5} {'std(q.k)':>9} {'sqrt(d_k)':>9} | tanpa skala: {'max w':>6} {'||J||':>7} | dibagi sqrt(d_k): {'max w':>6} {'||J||':>7}")
for dk in [2, 16, 64, 256, 768, 4096]:
    q_r = torch.randn(N_q, 1, dk, generator=uth_gen)
    k_r = torch.randn(N_q, T_tok, dk, generator=uth_gen)
    s_r = (q_r @ k_r.transpose(1, 2)).squeeze(1)                    # (N_q, T_tok) skor mentah
    row = [dk, s_r.std().item(), dk ** 0.5]
    for scale in (1.0, dk ** 0.5):
        p_r = torch.softmax(s_r / scale, dim=-1)
        J_r = torch.diag_embed(p_r) - p_r.unsqueeze(-1) * p_r.unsqueeze(-2)   # dp/dlogit, logit = s / scale
        row += [p_r.max(dim=-1).values.mean().item(), J_r.flatten(1).norm(dim=-1).mean().item()]
    uth1.append(row)
    print(f"{row[0]:>5} {row[1]:>9.3f} {row[2]:>9.3f} | {'':>12} {row[3]:>6.4f} {row[4]:>7.4f} |"
          f" {'':>17} {row[5]:>6.4f} {row[6]:>7.4f}")


  d_k  std(q.k) sqrt(d_k) | tanpa skala:  max w   ||J|| | dibagi sqrt(d_k):  max w   ||J||
    2     1.396     1.414 |              0.4580  0.3684 |                   0.3822  0.3800
   16     4.065     4.000 |              0.7715  0.2436 |                   0.4170  0.3844
   64     8.078     8.000 |              0.8885  0.1379 |                   0.4217  0.3849
  256    16.077    16.000 |              0.9449  0.0746 |                   0.4179  0.3865
  768    27.517    27.713 |              0.9653  0.0488 |                   0.4165  0.3857
 4096    64.894    64.000 |              0.9860  0.0197 |                   0.4250  0.3848


Ketiga mata rantai klaim buku terbukti, masing-masing dengan angka:

1. **Skor membesar seperti $\sqrt{d_k}$.** Std skor terukur 8.08 di $d_k = 64$ dan 27.52 di $d_k = 768$, praktis sama dengan $\sqrt{d_k}$ (8 dan 27.7). Ini cocok dengan varians $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$, yaitu $d_k$.
2. **Softmax menjadi hampir fungsi tangga.** Tanpa skala, rata-rata bobot terbesar naik dari 0.4580 ($d_k = 2$) ke 0.9860 ($d_k = 4096$). Hampir seluruh massa jatuh ke satu token.
3. **Softmax tersaturasi, gradien lewat softmax mengecil.** Tanpa skala, logit adalah skor mentah, sehingga $\|J\|$ di sini sekaligus turunan terhadap $s$. Nilainya turun dari 0.3684 ke 0.0197, sekitar 19x lebih kecil. Setelah dibagi $\sqrt{d_k}$, std logit kembali ke sekitar 1 dan $\|J\|$ terhadap logit $s/\sqrt{d_k}$ bertahan di 0.380-0.386 untuk *semua* $d_k$. Bobot terbesarnya pun konstan di sekitar 0.42. Artinya softmax tidak pernah tersaturasi.

Ada satu nuansa yang baru relevan di 3.6. Di `MultiHeadAttention`, `keys.shape[-1]` adalah `head_dim`, bukan lebar embedding. Untuk GPT-2 small nilainya $768 / 12 = 64$, jadi pembaginya $\sqrt{64} = 8$. Di baris $d_k = 64$, softmax tanpa skala sudah cukup tersaturasi: bobot terbesar 0.8885 dan $\|J\|$ terhadap logit 0.1379. Versi berskala punya bobot terbesar 0.4217 dan $\|J\|$ 0.3849. Kedua angka $\|J\|$ itu diukur terhadap logit masing-masing, jadi rasionya *bukan* rasio gradien terhadap skor mentah. Terhadap $s$, turunan versi berskala adalah $J/\sqrt{64}$ = 0.3849/8 = 0.0481. Yang dicegah oleh skala adalah saturasi, yaitu distribusi yang hampir one-hot dan bergantung pada input. Faktor $1/\sqrt{d_k}$ sendiri adalah skala tetap yang sama untuk semua input.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.17.png" width="700">

**Figure 3.17** Langkah terakhir: value vector ditimbang dengan bobot attention lalu dijumlahkan menjadi context vector $z^{(2)}$.

In [25]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


$z^{(2)} = $ `[0.3061, 0.8210]`, cocok dengan buku (hlm. 70). Rumusnya sama dengan 3.3 (bobot @ vektor), tapi yang ditimbang sekarang adalah *value*. Lebar context vector ikut lebar value (`d_out = 2`), bukan lebar input.

### 3.4.2 Implementing a compact self-attention Python class

Langkah-langkah tadi dibungkus menjadi `nn.Module` supaya bisa dipasang ke arsitektur LLM di bab 4. Repo menaruh Listing 3.1 dan pemakaiannya di sel yang sama, sedangkan buku memisahkannya.

In [26]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Enam context vector berukuran 2, cocok dengan buku (hlm. 71). Baris kedua `[0.3061, 0.8210]` sama persis dengan `context_vec_2` di atas. Alasannya bukan kebetulan. Seed 123 diulang, dan `__init__` mengambil tiga `torch.rand(3, 2)` dengan urutan yang sama seperti 3.4.1 (query, key, value), jadi bobotnya identik. `grad_fn=<MmBackward0>` muncul karena bobot sekarang `nn.Parameter` dengan `requires_grad=True`.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.18.png" width="700">

**Figure 3.18** Self-attention dalam bentuk matriks: input $X$ dikalikan $W_q$, $W_k$, $W_v$ menjadi $Q$, $K$, $V$; matriks bobot attention dari $Q$ dan $K$ menimbang $V$ menjadi context vector $Z$.

In [27]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


`SelfAttention_v2` mengganti `nn.Parameter(torch.rand(...))` dengan `nn.Linear(bias=False)`, yang secara matematis sama dengan perkalian matriks. Outputnya cocok dengan buku (hlm. 73). Nilainya kecil, sekitar `-0.07` dan `0.07`, dan keenam barisnya hampir identik. Penyebabnya adalah init `nn.Linear` (Kaiming-uniform, batas $1/\sqrt{d_{in}} = 1/\sqrt{3} = 0.577$) yang simetris di sekitar nol. Bandingkan dengan `torch.rand` yang semuanya positif. Proyeksi yang kecil menghasilkan skor kecil, softmax yang hampir rata, dan context vector yang mendekati rata-rata value.

**Exercise 3.1** muncul di titik ini dalam buku: transplantasi bobot `SelfAttention_v2` ke `SelfAttention_v1`. Solusi repo mendefinisikan ulang `SelfAttention_v2` dan membuat ulang `sa_v2` dengan seed 123. Padahal 3.5.1 memakai `sa_v2` seed 789 dari sel di atas. Karena itu solusinya dijalankan di bagian *Exercises* di akhir notebook. Dengan begitu kernel yang dijalankan dari atas ke bawah tetap menghasilkan angka buku.

## 3.5 Hiding future words with causal attention

GPT menghasilkan teks dari kiri ke kanan. Saat memproses token ke-$t$, model tidak boleh melihat token sesudahnya. *Causal attention*, disebut juga masked attention, menutup semua bobot di atas diagonal matriks attention. Bobot yang tersisa kemudian dinormalisasi ulang supaya setiap baris tetap berjumlah 1 (figure 3.19).

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.19.png" width="700">

**Figure 3.19** Causal mask: bobot di atas diagonal ditutup, sehingga "journey" hanya melihat "Your" dan dirinya sendiri. Baris lalu dinormalisasi ulang.

### 3.5.1 Applying a causal attention mask

Rute pertama terdiri dari tiga langkah (figure 3.20): softmax, kalikan dengan mask nol-satu, lalu normalisasi ulang setiap baris. Untuk memudahkan, bobot query dan key diambil dari `sa_v2` (seed 789) di 3.4.2.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.20.png" width="700">

**Figure 3.20** Rute pertama: softmax atas skor, nolkan elemen di atas diagonal, lalu normalisasi ulang setiap baris.

In [28]:
# Reuse the query and key weight matrices of the
# SelfAttention_v2 object from the previous section for convenience
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


Matriks bobot $6 \times 6$ tanpa mask, cocok dengan buku (hlm. 75). Semua bobot berada di rentang 0.1477-0.2041, hampir rata ($1/6 = 0.1667$). Ini lanjutan dari proyeksi `sa_v2` yang kecil di 3.4.2. Kolom pertama ("Your") sedikit unggul di semua baris.

In [29]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


`torch.tril` membuat mask segitiga bawah. Angka 1 menandai posisi yang boleh dilihat, yaitu dirinya sendiri dan semua token sebelumnya. Baris $t$ berisi tepat $t$ buah angka 1.

In [30]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


Perkalian elemen menolkan semua posisi masa depan, tapi baris tidak lagi berjumlah 1. Baris pertama hanya menyisakan 0.1921. Baris terakhir masih utuh, karena token terakhir boleh melihat semuanya.

In [31]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


Setelah setiap baris dibagi jumlahnya, hasilnya cocok dengan buku (hlm. 76). Baris pertama `[1.0000, 0, ...]`: token pertama hanya bisa memperhatikan dirinya sendiri. Baris kedua `[0.5517, 0.4483]` sama dengan $0.2041 / (0.2041 + 0.1659)$ dan $0.1659 / 0.3700$. Rasio antarbobot yang tersisa tidak berubah oleh normalisasi ulang.

Rute kedua lebih ringkas (figure 3.21). Isi posisi masa depan dengan $-\infty$ *sebelum* softmax. Karena $e^{-\infty} = 0$, softmax langsung menghasilkan baris yang berjumlah 1.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.21.png" width="700">

**Figure 3.21** Rute kedua yang lebih efisien: isi skor di atas diagonal dengan $-\infty$, lalu softmax. Normalisasi ulang tidak perlu lagi.

In [32]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


Skor mentah dengan `-inf` di atas diagonal, cocok dengan buku (hlm. 77). Mask dibuat dengan `triu(..., diagonal=1)`, sehingga angka 1 menandai posisi yang *ditutup*. Konvensinya terbalik dari `mask_simple`. `masked_fill` (tanpa garis bawah) mengembalikan tensor baru, dan `attn_scores` tidak berubah.

In [33]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


Bobotnya sama dengan hasil rute pertama, dan cocok dengan buku (hlm. 78). Ada dua beda versi kecil di sini. Buku mencetak `dim=1` di sel ini dan `keys.shape[-1]` di 3.4.1. Repo memakai `dim=-1` dan `keys.shape[1]`. Sel berikut (kode orisinal) mengukur apakah perbedaan itu dan kedua rute benar-benar setara.

In [34]:
w_book = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)      # ejaan buku hlm. 78
print("softmax dim=1 (buku) == dim=-1 (repo) :", torch.equal(w_book, attn_weights))
print("keys.shape[1] == keys.shape[-1]       :", keys.shape[1] == keys.shape[-1], tuple(keys.shape))
print("rute 1 (renorm) vs rute 2 (-inf), max |selisih| :", (masked_simple_norm - attn_weights).abs().max().item())


softmax dim=1 (buku) == dim=-1 (repo) : True
keys.shape[1] == keys.shape[-1]       : True (6, 2)
rute 1 (renorm) vs rute 2 (-inf), max |selisih| : 2.9802322387695312e-08


Untuk tensor 2D, `dim=1` dan `dim=-1` menunjuk sumbu yang sama, dan hasilnya identik bit-per-bit. `keys.shape[1]` dan `keys.shape[-1]` sama-sama 2. Jadi beda ejaan buku dan repo bukan errata. Ejaan `dim=-1` tetap lebih aman, karena di Listing 3.3 tensor menjadi 3D dan `dim=1` akan menunjuk sumbu token. Kedua rute masking berselisih maksimal 2.98e-08. Angkanya sama sampai pembulatan float32, tapi jalur aritmetikanya berbeda, sehingga tidak dijamin bit-identik.

Buku menjelaskan dengan argumen aljabar bahwa normalisasi ulang setelah masking tidak membocorkan informasi masa depan. UTH 2 mengukurnya langsung.

### Under the hood 2 -- causal mask benar-benar menutup masa depan

`causal_context` adalah resep 3.5.1 (skor, $-\infty$, softmax berskala, rata-rata value) dengan bobot `sa_v2`. Sel ini menguji dua hal. Pertama, embedding token 4-6 ("with", "one", "step") diganggu dengan noise $10\sigma$, lalu context vector token 1-3 diperiksa apakah bergeser. Kedua, Jacobian penuh $\partial z_t / \partial x_s$ dihitung dengan autograd. Untuk causal attention, semua turunan terhadap token masa depan ($s > t$) harus tepat nol. Versi tanpa mask menjadi pembanding.

In [35]:
def causal_context(x, causal=True):
    q_c, k_c, v_c = sa_v2.W_query(x), sa_v2.W_key(x), sa_v2.W_value(x)
    s_c = q_c @ k_c.T
    if causal:
        n = x.shape[0]
        s_c = s_c.masked_fill(torch.triu(torch.ones(n, n), diagonal=1).bool(), -torch.inf)
    return torch.softmax(s_c / k_c.shape[-1]**0.5, dim=-1) @ v_c

x_future = inputs.clone()
x_future[3:] += 10 * torch.randn(3, 3, generator=torch.Generator().manual_seed(0))
z_base, z_pert = causal_context(inputs), causal_context(x_future)
uth2_shift = (z_base[3:] - z_pert[3:]).abs().max().item()
print("token 1-3 bit-identik setelah token 4-6 diganggu :", torch.equal(z_base[:3], z_pert[:3]))
print("token 4-6, max |selisih|                         :", round(uth2_shift, 4))

dep_causal = torch.autograd.functional.jacobian(causal_context, inputs).abs().sum(dim=(1, 3))
dep_full = torch.autograd.functional.jacobian(lambda x: causal_context(x, causal=False), inputs).abs().sum(dim=(1, 3))
print("sum |dz_t/dx_s|, baris t = output, kolom s = input (causal):")
print(dep_causal)
print("segitiga atas (s > t), max   :", dep_causal.triu(1).max().item())
print("tanpa mask, entri nol        :", int((dep_full == 0).sum()), "dari 36")


token 1-3 bit-identik setelah token 4-6 diganggu : True
token 4-6, max |selisih|                         : 12.5887
sum |dz_t/dx_s|, baris t = output, kolom s = input (causal):
tensor([[1.3988, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7712, 0.6285, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5307, 0.4332, 0.4362, 0.0000, 0.0000, 0.0000],
        [0.3857, 0.3443, 0.3449, 0.3236, 0.0000, 0.0000],
        [0.3035, 0.2768, 0.2771, 0.2632, 0.2781, 0.0000],
        [0.2699, 0.2323, 0.2330, 0.2147, 0.2389, 0.2041]])
segitiga atas (s > t), max   : 0.0
tanpa mask, entri nol        : 0 dari 36


Hasilnya tegas di kedua uji.

- **Gangguan.** Context vector token 1-3 tetap identik bit-per-bit (`torch.equal`), padahal token 4-6 diganggu sampai outputnya bergeser hingga 12.5887.
- **Jacobian.** Matriks sensitivitas berbentuk segitiga bawah murni. Nilai maksimum di atas diagonal adalah 0.0: nol eksak, bukan sekadar kecil. Ini terjadi karena $e^{-\infty} = 0$ memutus jalur gradien lewat key dan value masa depan sekaligus.
- **Pembanding.** Tanpa mask, 0 dari 36 entri bernilai nol, jadi setiap token bergantung pada semua token.

Dua pola lain terbaca dari matriks itu. Diagonal baris pertama paling besar (1.3988), karena token pertama hanya punya satu sumber. Semakin ke bawah, sensitivitas per sumber mengecil (baris terakhir 0.2041-0.2699). Penyebabnya, bobot dibagi ke lebih banyak token. Inilah jaminan yang membuat training next-token di bab 5 sah: prediksi untuk posisi $t$ tidak pernah melihat jawaban di posisi $t+1$.

### 3.5.2 Masking additional attention weights with dropout

Dropout menolkan unit secara acak saat training supaya model tidak bergantung pada unit tertentu, dan dimatikan saat inferensi. Di transformer, dropout biasanya dipasang setelah bobot attention dihitung (figure 3.22). Buku memakai rate 0.5 supaya efeknya terlihat jelas. GPT di bab-bab berikutnya memakai 0.1.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.22.png" width="700">

**Figure 3.22** Mask dropout acak diterapkan di atas bobot causal attention, sehingga sebagian bobot ikut nol saat training.

In [36]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


Mask ini cocok dengan buku (hlm. 79) elemen per elemen. Yang dinolkan adalah 15 dari 36 elemen (41.7%), bukan tepat setengah, karena setiap elemen diundi sendiri dengan peluang 0.5. Elemen yang selamat dikali $1/(1-0.5) = 2$ supaya nilai harapan setiap elemen tetap 1 ($0.5 \cdot 2 = 1$).

In [37]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


Dropout yang sama diterapkan ke bobot causal dari 3.5.1, dengan seed 123 yang sama. Hasilnya cocok dengan buku (hlm. 80). Ada dua cacat cetak di halaman itu: `0 .0000` yang terpisah spasi dan tanda `)` penutup yang hilang. Keduanya cacat tata letak, bukan perbedaan angka, dan ledger book-match menormalkan satu-satunya spasi itu.

Ada konsekuensi yang mudah terlewat. Baris kedua ("journey") nol seluruhnya, karena kedua bobotnya kebetulan didrop. Pada forward pass ini, context vector journey adalah vektor nol. Baris pertama bernilai 2.0, yaitu bobot tunggal 1.0 dikali 2. Jadi selama training, setiap baris bobot attention *tidak* lagi berjumlah 1. UTH 3 mengukur seberapa sering itu terjadi.

Buku dan repo mencatat bahwa output dropout bisa berbeda antar-sistem operasi, merujuk ke isu PyTorch [#121595](https://github.com/pytorch/pytorch/issues/121595). Mesin ini mereproduksi kedua mask buku persis. UTH 3 menunjukkan alasannya, sekaligus membedakan perbedaan RNG/backend dari bug kode.

### Under the hood 3 -- dropout: RNG, backend, train vs eval

Sel pertama adalah *fingerprint* dropout. Ia mengulang dropout buku (seed 123, rate 0.5, `ones(6, 6)`) lalu mengujinya terhadap dua acuan. Acuan pertama adalah mask buku hlm. 79. Acuan kedua adalah rekonstruksi dari undian uniform biasa: elemen dipertahankan jika $u < 0.5$, dengan $u$ dari `torch.rand(..., dtype=float64)` pada seed yang sama. Kalau rekonstruksi cocok, mask dropout di mesin ini tidak lain adalah undian uniform float64 dari generator CPU. Source sel ini identik byte-per-byte dengan `replay/dropout_probe.py` di folder build.

In [38]:
import platform
import torch

BOOK_MASK = torch.tensor([[2., 2., 0., 2., 2., 0.],
                          [0., 0., 0., 2., 0., 2.],
                          [2., 2., 2., 2., 0., 2.],
                          [0., 2., 2., 0., 0., 2.],
                          [0., 2., 0., 2., 0., 2.],
                          [0., 2., 2., 2., 2., 0.]])


def dropout_fingerprint():
    torch.manual_seed(123)
    out = torch.nn.Dropout(0.5)(torch.ones(6, 6))
    torch.manual_seed(123)
    u = torch.rand(6, 6, dtype=torch.float64)
    print(f"platform               : {platform.system()} {platform.machine()} | torch {torch.__version__}"
          f" | MKL {torch.backends.mkl.is_available()}")
    print(f"mask == book (p. 79)   : {torch.equal(out, BOOK_MASK)}")
    print(f"mask == 2*(u64 < 0.5)  : {torch.equal(out, (u < 0.5).float() * 2)}")
    print(f"zeros                  : {int((out == 0).sum())}/36 | row 0: {out[0].tolist()}")


dropout_fingerprint()

platform               : Linux x86_64 | torch 2.11.0+cu130 | MKL True
mask == book (p. 79)   : True
mask == 2*(u64 < 0.5)  : True
zeros                  : 15/36 | row 0: [2.0, 2.0, 0.0, 2.0, 2.0, 0.0]


Di `Darwin arm64 | torch 2.14.1 | MKL False`, mask dropout identik dengan mask buku dan identik dengan `2 * (u64 < 0.5)`. Jadi `Dropout` di build ini menarik satu angka uniform float64 per elemen dari Mersenne Twister generator CPU dan membandingkannya dengan $1 - p$. Build torch ini dikompilasi tanpa MKL (`MKL False`).

Kesimpulan yang sah dari sel ini hanya untuk konfigurasi yang diuji: macOS arm64, torch 2.14.1, tanpa MKL. Build lain bisa memakai jalur RNG berbeda untuk `bernoulli_`. Kalau itu terjadi, mask akan berbeda walaupun kodenya identik. Perbedaan seperti itu adalah perbedaan RNG/backend, bukan bug kode.

Sel kedua memeriksa perilaku dropout yang memengaruhi training. Isinya: jumlah baris bobot causal setelah dropout dalam 10,000 undian, peluang satu baris nol seluruhnya, rate efektif pada tensor besar, apakah `torch.no_grad()` mematikan dropout, apa efek `eval()`, dan mask di akselerator.

In [39]:
torch.manual_seed(123)
drop = torch.nn.Dropout(0.5)
w_causal = attn_weights.detach()
trials = torch.stack([drop(w_causal) for _ in range(10_000)])
rs = trials.sum(dim=-1)
print("rowsum, undian pertama       :", rs[0])
print("rowsum, rata-rata 10k undian :", rs.mean(dim=0))
print("P(baris nol semua), terukur  :", (rs == 0).float().mean(dim=0))
print("P(baris nol semua), 0.5^t    :", 0.5 ** torch.arange(1, 7))
big = drop(torch.ones(1000, 1000))
print(f"ones(1000, 1000): frac nol {(big == 0).float().mean().item():.4f} | mean {big.mean().item():.4f}")
with torch.no_grad():
    print("no_grad + mode train, masih ada nol?   :", bool((drop(torch.ones(6, 6)) == 0).any()))
drop.eval()
print("mode eval, output == input (identitas)?:", torch.equal(drop(w_causal), w_causal))
drop.train()

if device.type == "cpu":
    print("akselerator: device = cpu, perbandingan backend dilewati")
else:
    torch.manual_seed(123)
    m_acc = torch.nn.Dropout(0.5)(torch.ones(6, 6, device=device)).cpu()
    print(f"akselerator {device.type}: mask == mask buku? {torch.equal(m_acc, BOOK_MASK)} | nol {int((m_acc == 0).sum())}/36")


rowsum, undian pertama       : tensor([2.0000, 0.0000, 2.0000, 0.9846, 0.7741, 1.3073])
rowsum, rata-rata 10k undian : tensor([1.0050, 0.9972, 0.9950, 1.0003, 1.0017, 1.0027])
P(baris nol semua), terukur  : tensor([0.4975, 0.2546, 0.1258, 0.0625, 0.0313, 0.0147])
P(baris nol semua), 0.5^t    : tensor([0.5000, 0.2500, 0.1250, 0.0625, 0.0312, 0.0156])
ones(1000, 1000): frac nol 0.5001 | mean 0.9998
no_grad + mode train, masih ada nol?   : True
mode eval, output == input (identitas)?: True
akselerator cuda: mask == mask buku? False | nol 14/36


Dari sel ini terbaca beberapa hal.

- **Rowsum saat train bukan 1.** Pada undian pertama, jumlah keenam baris adalah 2.0000, 0.0000, 2.0000, 0.9846, 0.7741, 1.3073. Ada baris yang menjadi 0 dan ada yang melampaui 1. Hanya *rata-ratanya* yang tetap 1: dalam 10,000 undian, rata-rata per baris adalah 1.0050, 0.9972, 0.9950, 1.0003, 1.0017, 1.0027. Dropout invertible menjaga nilai harapan, bukan normalisasi per sampel.
- **Baris nol seluruhnya.** Peluang terukurnya 0.4975, 0.2546, 0.1258, 0.0625, 0.0313, 0.0147, cocok dengan $0.5^t$, karena baris $t$ punya tepat $t$ bobot tak-nol. Token pertama kehilangan seluruh konteksnya di sekitar separuh forward pass. Itu konsekuensi wajar rate 0.5 di attention, dan salah satu alasan GPT memakai 0.1.
- **Rate efektif.** Pada `ones(1000, 1000)`, fraksi nol adalah 0.5001 dan mean 0.9998. Angka 15/36 di sel buku hanyalah variasi sampel kecil.
- **`no_grad` bukan `eval`.** Di dalam `torch.no_grad()` dengan modul tetap di mode train, dropout *masih* menolkan elemen. Hanya `.eval()` yang mematikannya, dan outputnya identik dengan input. Kalau evaluasi di bab 5 lupa memanggil `model.eval()`, loss validasi ikut teracak dropout.
- **Backend.** Di MPS dengan seed yang sama, mask berbeda dari mask buku (14/36 nol). `torch.manual_seed` memang menyeed semua device, tapi setiap backend punya generator dan kernel dropout sendiri. Kode yang sama menghasilkan mask yang sah namun berbeda. Sel ini bergantung pada device: di Colab dengan GPU baris ini memakai CUDA, dan di mesin CPU-only baris ini dilewati.

Pemisahan yang dipakai notebook ini: mask yang berbeda karena RNG atau backend (perangkat, build library) bukan errata kode. Errata kode harus berupa kode yang salah atau output tercetak yang tidak bisa dihasilkan oleh kodenya sendiri. Contoh kategori kedua ada di Exercise 3.2.

### 3.5.3 Implementing a compact causal attention class

Kelas causal attention harus menerima *batch* seperti keluaran data loader bab 2: tensor `(batch, num_tokens, d_in)`. Untuk simulasi, kalimat contoh diduplikasi menjadi batch berisi dua.

In [40]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

torch.Size([2, 6, 3])


Shape `(2, 6, 3)`: dua teks, enam token, embedding 3 dimensi. Cocok dengan buku (hlm. 80).

In [41]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Listing 3.3 menggabungkan semua yang sudah dibangun: proyeksi `nn.Linear` dari `SelfAttention_v2`, mask $-\infty$, dan dropout. Ada empat perubahan yang perlu dicatat.

1. `keys.transpose(1, 2)` menggantikan `.T`, supaya dimensi batch tetap di posisi 0.
2. Mask disimpan dengan `register_buffer`. Buffer ikut pindah device bersama model, tapi tidak dihitung sebagai parameter.
3. `masked_fill_` bekerja in-place.
4. Mask dipotong ke `[:num_tokens, :num_tokens]`, sehingga sekuens yang lebih pendek dari `context_length` tetap bisa diproses.

Buku hanya mencetak shape `(2, 6, 2)`, dan shape itu cocok (hlm. 82). Repo juga mencetak tensornya. Kedua batch identik karena inputnya duplikat. `grad_fn=<UnsafeViewBackward0>` muncul karena matmul 3D diratakan secara internal. Catatan teks: anotasi buku menyebut kelas ini sebagai pengembangan dari `SelfAttention_v1`. Source di atas memakai `nn.Linear`, jadi turunannya adalah `SelfAttention_v2`.

Komentar di listing repo menyatakan bahwa input yang lebih panjang dari `context_length` akan error di pembuatan mask. Sel berikut (orisinal) menguji klaim itu dengan 7 token ke modul yang dibuat untuk 6.

In [42]:
ca(torch.ones(1, 7, 3))

RuntimeError: The size of tensor a (7) must match the size of tensor b (6) at non-singleton dimension 2

Klaim komentar repo terbukti. Error-nya bukan di pembuatan mask. Slicing `self.mask[:7, :7]` pada buffer $6 \times 6$ diam-diam mengembalikan $6 \times 6$, lalu broadcasting `masked_fill_` gagal terhadap skor $7 \times 7$. Buffer mask berukuran tetap, sehingga `context_length` adalah batas keras panjang input. Bab 4-7 menjaga batas ini dengan memotong input ke `context_length` sebelum forward. Ini sel error yang disengaja (`raises-exception`), bukan errata.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.23.png" width="700">

**Figure 3.23** Posisi di peta bab: simplified self-attention, self-attention dengan bobot latih, dan causal attention sudah selesai. Multi-head attention berikutnya.

## 3.6 Extending single-head attention to multi-head attention

### 3.6.1 Stacking multiple single-head attention layers

Multi-head attention menjalankan beberapa causal attention secara paralel, masing-masing dengan proyeksi $W_q, W_k, W_v$ sendiri, lalu menggabungkan output-nya (figure 3.24). Versi paling langsung adalah membuat beberapa `CausalAttention` dalam `nn.ModuleList` lalu menyambung output-nya di dimensi terakhir.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.24.png" width="700">

**Figure 3.24** Multi-head attention dengan dua head: dua set matriks bobot menghasilkan dua set context vector $Z_1$ dan $Z_2$, yang digabung menjadi $Z$.

In [43]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Output `(2, 6, 4)`, cocok dengan buku (hlm. 85). Dua head dengan `d_out = 2` masing-masing menghasilkan $2 \times 2 = 4$ kolom. Dua kolom pertama identik dengan output `CausalAttention` di 3.5.3 (`[-0.4519, 0.2216]` dan seterusnya), dan itu bisa diprediksi. Seed 123 diulang, lalu head pertama dibuat lebih dulu, sehingga ia mengambil bobot yang persis sama dengan `ca`. Kolom 3-4 berasal dari head kedua, yang bobotnya diambil dari aliran RNG berikutnya.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.25.png" width="700">

**Figure 3.25** `MultiHeadAttentionWrapper` dengan `num_heads=2` dan `d_out=2` per head: dua matriks context vector disambung di dimensi kolom menjadi lebar 4.

**Exercise 3.2** (buku) meminta output 2 dimensi dengan `num_heads=2` tetap. Solusinya dijalankan di bagian *Exercises*. Hasilnya ternyata membuka satu errata pada output tercetak di solusi repo.

### 3.6.2 Implementing multi-head attention with weight splits

Wrapper menjalankan proyeksi dan attention setiap head secara berurutan, dengan `[head(x) for head in self.heads]`. `MultiHeadAttention` mengganti pola itu dengan satu proyeksi besar per Q/K/V. Hasil proyeksi lalu dipecah menjadi head lewat `.view(b, num_tokens, num_heads, head_dim)` dan `.transpose(1, 2)`. Attention semua head dihitung dalam satu batched matmul, head digabung kembali, lalu lewat `out_proj` (figure 3.26). Repo menaruh Listing 3.5 dan pemakaiannya di satu sel.

In [44]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Output `(2, 6, 2)`, cocok dengan buku (hlm. 90). Di kelas ini `d_out` adalah lebar *total*, sehingga `d_out=2` dengan 2 head berarti `head_dim = 1`. Angkanya berbeda dari wrapper karena bobot diambil dalam bentuk dan urutan berbeda, dan ada `out_proj` tambahan. Kesetaraan kedua implementasi diuji langsung di UTH 4. `grad_fn=<ViewBackward0>` berasal dari `out_proj` yang diterapkan ke tensor 3D.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.26.png" width="700">

**Figure 3.26** Atas: wrapper dengan dua matriks query terpisah $W_{q1}$ dan $W_{q2}$. Bawah: `MultiHeadAttention` dengan satu $W_q$ besar, lalu $Q$ dipecah menjadi $Q_1$ dan $Q_2$. Key dan value diperlakukan sama.

Supaya batched matmul 4D tidak terasa ajaib, buku memakai tensor kecil berbentuk `(b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)`.

In [45]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


`a @ a.transpose(2, 3)` mengalikan dua dimensi terakhir, `(3, 4) @ (4, 3)`, untuk setiap pasangan batch dan head. Hasilnya `(1, 2, 3, 3)`, cocok dengan buku (hlm. 89). Setiap matriks $3 \times 3$ simetris, karena perkalian ini adalah $A A^\top$ per head.

In [46]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


Perkalian manual per head memberi angka yang sama persis dengan versi batched. Inilah yang terjadi di dalam `MultiHeadAttention`: satu operator `@` menggantikan loop atas head.

Buku mengklaim `MultiHeadAttention` lebih efisien karena proyeksi Q/K/V hanya dihitung sekali untuk semua head. UTH 4 mengukur klaim efisiensi itu, sekaligus klaim bahwa kedua kelas mengimplementasikan konsep yang sama.

### Under the hood 4 -- wrapper vs `MultiHeadAttention`: sama secara matematis, beda biaya

Kesetaraan diuji dengan *transplantasi bobot*. Bobot Q/K/V dari kedua head wrapper disalin berurutan ke matriks tunggal `MultiHeadAttention` (baris $0..h-1$ untuk head 1, dan seterusnya). `out_proj` diset menjadi identitas. Jika kedua kelas memang implementasi yang sama, output-nya akan sama sampai pembulatan float. Sel yang sama menghitung parameter dan buffer pada skala GPT-2 small, lalu membandingkan `MultiHeadAttention` dengan kernel fused `F.scaled_dot_product_attention` milik PyTorch.

In [47]:
import time
import torch.nn.functional as F

torch.manual_seed(123)
wrap_t = MultiHeadAttentionWrapper(3, 2, 6, 0.0, num_heads=2)
mha_t = MultiHeadAttention(3, 4, 6, 0.0, num_heads=2)
with torch.no_grad():
    for name in ["W_query", "W_key", "W_value"]:
        getattr(mha_t, name).weight.copy_(torch.cat([getattr(h, name).weight for h in wrap_t.heads], dim=0))
    mha_t.out_proj.weight.copy_(torch.eye(4))
    mha_t.out_proj.bias.zero_()
    uth4_diff = (wrap_t(batch) - mha_t(batch)).abs().max().item()
print(f"transplantasi bobot, max |wrapper - MHA| : {uth4_diff:.2e}")

torch.manual_seed(123)
d_g, H_g, T_g = 768, 12, 1024
wrap_g = MultiHeadAttentionWrapper(d_g, d_g // H_g, T_g, 0.0, num_heads=H_g).eval()
mha_g = MultiHeadAttention(d_g, d_g, T_g, 0.0, num_heads=H_g).eval()
x_g = torch.randn(2, T_g, d_g)

def mha_sdpa(x):
    b, n, _ = x.shape
    split = lambda t: t.view(b, n, H_g, d_g // H_g).transpose(1, 2)
    q, k, v = split(mha_g.W_query(x)), split(mha_g.W_key(x)), split(mha_g.W_value(x))
    o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return mha_g.out_proj(o.transpose(1, 2).contiguous().view(b, n, d_g))

with torch.no_grad():
    uth4_sdpa = (mha_sdpa(x_g) - mha_g(x_g)).abs().max().item()
n_par = lambda m: sum(p.numel() for p in m.parameters())
n_buf = lambda m: sum(b.numel() for b in m.buffers())
print(f"parameter : wrapper {n_par(wrap_g):,} | MHA {n_par(mha_g):,} (out_proj {n_par(mha_g.out_proj):,})")
print(f"buffer    : wrapper {n_buf(wrap_g):,} | MHA {n_buf(mha_g):,} elemen mask")
print(f"proyeksi  : wrapper {3 * H_g} Linear kecil (768 -> 64) | MHA 3 Linear besar (768 -> 768) + out_proj")
print(f"SDPA (is_causal=True) vs MHA, max |selisih| : {uth4_sdpa:.2e}")


transplantasi bobot, max |wrapper - MHA| : 5.96e-08
parameter : wrapper 1,769,472 | MHA 2,360,064 (out_proj 590,592)
buffer    : wrapper 12,582,912 | MHA 1,048,576 elemen mask
proyeksi  : wrapper 36 Linear kecil (768 -> 64) | MHA 3 Linear besar (768 -> 768) + out_proj
SDPA (is_causal=True) vs MHA, max |selisih| : 1.94e-07


Hasil yang deterministik:

- **Sama secara matematis.** Setelah transplantasi, wrapper dan `MultiHeadAttention` berselisih maksimal 5.96e-08, setara pembulatan float32. Selisihnya tidak nol, karena perkalian $(3 \times 4)$ sekaligus dan dua perkalian $(3 \times 2)$ menjumlahkan dengan urutan berbeda. Kedua kelas memang mengimplementasikan fungsi yang sama.
- **Parameter.** Wrapper GPT-2-size punya 1,769,472 parameter. `MultiHeadAttention` punya 2,360,064, dan selisihnya (590,592) persis `out_proj`. Proyeksi Q/K/V di kedua kelas berukuran sama ($3 \times 768^2$). Yang berbeda hanya cara memotongnya.
- **Buffer.** Wrapper menyimpan 12,582,912 elemen mask, yaitu 12 salinan identik mask $1024 \times 1024$ (sekitar 50 MB float32). `MultiHeadAttention` cukup satu salinan (1,048,576).
- **Kernel fused.** `F.scaled_dot_product_attention(..., is_causal=True)` dengan bobot yang sama berselisih maksimal 2.38e-07 dari `MultiHeadAttention`. Kernel PyTorch menghitung fungsi yang sama dengan Listing 3.5.

Sel berikut mengukur latency. Hasilnya nondeterministik, jadi dikecualikan dari pembandingan bit-per-bit di verifikasi kernel.

In [48]:
def median_ms(fn, reps=15):
    ts = []
    with torch.no_grad():
        fn(x_g)
        for _ in range(reps):
            t0 = time.perf_counter()
            fn(x_g)
            ts.append((time.perf_counter() - t0) * 1e3)
    return sorted(ts)[reps // 2]

uth4 = {"wrap": median_ms(wrap_g), "mha": median_ms(mha_g), "sdpa": median_ms(mha_sdpa)}
print(f"latency CPU, median 15 run, batch 2 x {T_g} token, {torch.get_num_threads()} thread:")
print(f"  wrapper (12 x CausalAttention) : {uth4['wrap']:7.1f} ms")
print(f"  MultiHeadAttention             : {uth4['mha']:7.1f} ms  ({uth4['wrap'] / uth4['mha']:.2f}x lebih cepat)")
print(f"  MHA + SDPA fused               : {uth4['sdpa']:7.1f} ms  ({uth4['wrap'] / uth4['sdpa']:.2f}x lebih cepat)")


latency CPU, median 15 run, batch 2 x 1024 token, 1 thread:
  wrapper (12 x CausalAttention) :   227.6 ms
  MultiHeadAttention             :   380.2 ms  (0.60x lebih cepat)
  MHA + SDPA fused               :   134.9 ms  (1.69x lebih cepat)


Pada build ini wrapper butuh 87.5 ms, `MultiHeadAttention` 52.2 ms (1.67x lebih cepat), dan versi SDPA 18.2 ms (4.81x). Klaim efisiensi buku terbukti dari urutannya, dengan catatan sumbernya. Proyeksi Q/K/V dan perkalian attention di kedua kelas sama besarnya, sekitar 13.69 GFLOP untuk batch ini (matmul skor dihitung penuh $T \times T$; mask tidak melewati kerja). `MultiHeadAttention` justru mengerjakan *lebih banyak*: `out_proj` dengan 590,592 parameter, yaitu satu matmul $768 \times 768$ per token, menambah sekitar 2.42 GFLOP (+17.6%). Jalur SDPA di atas juga memakai `out_proj` yang sama. Timing di sini valid untuk implementasi persis yang diukur. Keuntungan `MultiHeadAttention` datang dari bentuk kerjanya, bukan dari jumlah kerjanya: 3 matmul besar menggantikan 36 matmul kecil, dan satu batched matmul attention menggantikan loop Python atas 12 head. Pekerjaan sebesar itu lebih efisien bagi BLAS. Besar rasionya bergantung pada mesin, jumlah thread, dan beban saat diukur, jadi yang menjadi temuan adalah urutannya. SDPA unggul paling jauh karena mask, softmax, dan perkalian value digabung dalam satu kernel, tanpa mematerialisasi matriks skor $2 \times 12 \times 1024 \times 1024$ (sekitar 100 MB float32).

### Ukuran GPT-2 yang disebut buku

Bab ini ditutup dengan perbandingan skala. Buku menyebut GPT-2 terkecil memiliki **117 juta** parameter, 12 head, dan embedding 768. GPT-2 terbesar memiliki 1.5 miliar parameter, 25 head, dan embedding 1,600. Klaim ini diverifikasi dengan dua cara. Cara pertama: aritmetika dari arsitektur (tanpa jaringan), memakai `MultiHeadAttention` bab ini untuk bagian attention. MLP dan LayerNorm mengikuti arsitektur bab 4.

In [49]:
def gpt2_params(d, n_layers, n_heads, vocab=50257, ctx=1024):
    attn = MultiHeadAttention(d, d, ctx, 0.0, num_heads=n_heads, qkv_bias=True)
    p_attn = sum(p.numel() for p in attn.parameters())
    p_mlp = (d * 4 * d + 4 * d) + (4 * d * d + d)
    p_ln = 2 * (2 * d)
    total = vocab * d + ctx * d + n_layers * (p_attn + p_mlp + p_ln) + 2 * d
    return p_attn, total

for name, d, L, H in [("gpt2 (small)", 768, 12, 12), ("gpt2-xl", 1600, 48, 25)]:
    p_attn, total = gpt2_params(d, L, H)
    print(f"{name:13s} attention/blok {p_attn:>10,} | total {total:>13,}")


gpt2 (small)  attention/blok  2,362,368 | total   124,439,808
gpt2-xl       attention/blok 10,246,400 | total 1,557,611,200


Aritmetikanya menyusun GPT-2 dari komponen-komponen berikut:

- embedding token $50257 \times d$ dan embedding posisi $1024 \times d$;
- per blok: attention dengan bias QKV, karena GPT-2 asli memakai `qkv_bias=True`;
- per blok: MLP dengan ekspansi 4x;
- per blok: dua LayerNorm, masing-masing dengan scale dan shift;
- satu LayerNorm final.

Output head tidak dihitung terpisah karena GPT-2 berbagi bobot dengan embedding token. Totalnya **124,439,808** untuk GPT-2 small, bukan 117 juta, dan 1,557,611,200 untuk GPT-2 XL. Cara kedua memakai checkpoint resmi di Hugging Face: hanya header safetensors (sekitar 14 KB) plus satu tensor mask 4 MB yang diunduh lewat HTTP range request. Sel ini butuh jaringan.

In [50]:
import json
import math
import struct
import urllib.request

def hf_get(repo, fname, start=None, end=None):
    url = f"https://huggingface.co/{repo}/resolve/main/{fname}"
    headers = {"Range": f"bytes={start}-{end}"} if start is not None else {}
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
        return r.read()

hf = {}
for repo in ["openai-community/gpt2", "openai-community/gpt2-xl"]:
    cfg = json.loads(hf_get(repo, "config.json"))
    n_hdr = struct.unpack("<Q", hf_get(repo, "model.safetensors", 0, 7))[0]
    hdr = json.loads(hf_get(repo, "model.safetensors", 8, 7 + n_hdr))
    hdr.pop("__metadata__", None)
    numel = {k: math.prod(v["shape"]) for k, v in hdr.items()}
    buf = sum(n for k, n in numel.items() if k.endswith(".attn.bias"))
    hf[repo] = {"hdr": hdr, "n_hdr": n_hdr, "all": sum(numel.values()), "buf": buf}
    print(f"{repo}: n_head={cfg['n_head']} n_embd={cfg['n_embd']} n_layer={cfg['n_layer']}")
    print(f"    semua tensor {sum(numel.values()):>13,} = parameter {sum(numel.values()) - buf:>13,} + buffer mask {buf:>11,}")

g = hf["openai-community/gpt2"]
start, end = g["hdr"]["h.0.attn.bias"]["data_offsets"]
raw = hf_get("openai-community/gpt2", "model.safetensors", 8 + g["n_hdr"] + start, 8 + g["n_hdr"] + end - 1)
gpt2_mask = torch.frombuffer(bytearray(raw), dtype=torch.float32).view(1024, 1024)
print("h.0.attn.bias:", g["hdr"]["h.0.attn.bias"]["dtype"], g["hdr"]["h.0.attn.bias"]["shape"],
      "| == tril(ones):", torch.equal(gpt2_mask, torch.tril(torch.ones(1024, 1024))),
      "| == NOT mask bab ini:", torch.equal(gpt2_mask.bool(), ~torch.triu(torch.ones(1024, 1024), diagonal=1).bool()))


openai-community/gpt2: n_head=12 n_embd=768 n_layer=12
    semua tensor   137,022,720 = parameter   124,439,808 + buffer mask  12,582,912
openai-community/gpt2-xl: n_head=25 n_embd=1600 n_layer=48
    semua tensor 1,607,942,848 = parameter 1,557,611,200 + buffer mask  50,331,648
h.0.attn.bias: F32 [1, 1, 1024, 1024] | == tril(ones): True | == NOT mask bab ini: True


**Errata #1 -- GPT-2 terkecil berisi 124 juta parameter, bukan 117 juta.** Checkpoint resmi `openai-community/gpt2` memuat 124,439,808 parameter, persis sama dengan hasil aritmetika di atas. Angka 117M berasal dari tabel paper GPT-2 (Radford dkk., 2019). Bab 4 buku sendiri menamai konfigurasinya `GPT_CONFIG_124M`. Solusi Exercise 3.3 di repo mengulang angka 117M. Klaim lainnya terbukti: 12 head dan embedding 768 untuk small, 25 head dan embedding 1,600 untuk XL. Angka 1.5 miliar adalah pembulatan wajar dari 1,557,611,200.

Header checkpoint juga menunjukkan jebakan yang langsung berkaitan dengan `register_buffer` di 3.5.3. File itu menyimpan 137,022,720 angka, karena setiap blok membawa buffer `attn.bias` berupa mask $1024 \times 1024$ (total 12,582,912 elemen). Siapa pun yang menjumlahkan semua tensor di file akan mendapat 137 juta. Isi buffer itu adalah `tril(ones)`, dengan 1 berarti *boleh dilihat*. Itu komplemen persis dari mask `triu(..., diagonal=1)` bab ini, yang memakai 1 untuk *ditutup*. Konsepnya sama, konvensinya terbalik. Buffer itu juga menegaskan bahwa GPT-2 asli menyimpan mask kausal sebagai konstanta 4 MB per blok, bukan sebagai parameter yang dilatih.

## Exercises

Solusi diambil dari `exercise-solutions.ipynb` di repo. Exercise 3.1 tersedia sebagai code cell. Untuk Exercise 3.2 dan 3.3, repo hanya menyimpan blok kode di sel markdown, dan di sini blok itu dieksekusi verbatim untuk pertama kalinya. Semua solusi diletakkan di akhir karena solusi 3.1 mendefinisikan ulang `SelfAttention_v1`, `SelfAttention_v2`, dan `sa_v2`. Sel versi dari notebook solusi ikut disertakan.

In [51]:
from importlib.metadata import version

import torch
print("torch version:", version("torch"))

torch version: 2.11.0+cu130


Versi yang sama dengan sel setup. Notebook solusi repo juga menyimpan output torch 2.4.0.

### Exercise 3.1 -- Comparing `SelfAttention_v1` and `SelfAttention_v2`

Soalnya: salin bobot dari instance `SelfAttention_v2` ke instance `SelfAttention_v1` sehingga keduanya menghasilkan output yang sama. Kuncinya ada di cara `nn.Linear` menyimpan bobot. Atributnya `weight` berbentuk `(d_out, d_in)` dan forward menghitung $x W^\top$. `SelfAttention_v1` menyimpan matriks `(d_in, d_out)` dan menghitung $x W$. Empat sel berikut menyiapkan input, kedua kelas, dan transplantasinya.

In [52]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in, d_out = 3, 2

In [53]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

In [54]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)

In [55]:
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

Transplantasinya hanya `.T` pada setiap `weight`, dibungkus `nn.Parameter` baru. Solusi repo memakai seed 123 untuk kedua objek, dan `SelfAttention_v2` versi solusi memakai softmax `dim=1`. Untuk tensor 2D itu ekuivalen dengan `dim=-1`, seperti terukur di 3.5.1.

In [56]:
sa_v1(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

Output `sa_v1` setelah transplantasi adalah `[-0.5337, -0.1051]` dan seterusnya, sama dengan output tersimpan di repo.

In [57]:
sa_v2(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

`sa_v2` mencetak angka yang sama hingga empat desimal. Sel berikut (orisinal) memeriksa apakah kesamaan itu bit-exact.

In [58]:
out_v1, out_v2 = sa_v1(inputs), sa_v2(inputs)
print("bit-identik       :", torch.equal(out_v1, out_v2))
print("max |v1 - v2|     :", (out_v1 - out_v2).abs().max().item())
print("W_query v1 == W_q.weight.T dari v2:", torch.equal(sa_v1.W_query, sa_v2.W_query.weight.T))


bit-identik       : True
max |v1 - v2|     : 0.0
W_query v1 == W_q.weight.T dari v2: True


Output kedua kelas identik bit-per-bit. Bobot hasil transplantasi identik dengan `weight.T`. Jadi satu-satunya perbedaan `SelfAttention_v1` dan `SelfAttention_v2` adalah inisialisasi bobot (`torch.rand` vs Kaiming-uniform `nn.Linear`), persis seperti klaim buku.

### Exercise 3.2 -- Returning two-dimensional embedding vectors

Soalnya: dengan `num_heads=2` tetap, ubah *satu* argumen `MultiHeadAttentionWrapper` supaya output-nya 2 dimensi, bukan 4. Jawabannya `d_out=1` per head, sehingga $2 \times 1 = 2$ kolom. Kode di bawah adalah blok solusi repo, dieksekusi verbatim.

In [59]:
torch.manual_seed(123)

d_out = 1
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Shape-nya `(2, 6, 2)`, jadi solusi repo menjawab soal dengan benar. Baris pertama batch adalah `[-0.5740, 0.2216]`. Repo tidak pernah mengeksekusi blok ini dan menyimpan output tercetak di sel markdown sebelahnya, dan angka di sana *berbeda* dari run ini. Sel berikut (orisinal) menguji apakah selisih itu bisa dijelaskan RNG/backend atau tidak.

In [60]:
repo_claim = torch.tensor([[-9.1476e-02,  3.4164e-02],
                           [-2.6796e-01, -1.3427e-03],
                           [-4.8421e-01, -4.8909e-02],
                           [-6.4808e-01, -1.0625e-01],
                           [-8.8380e-01, -1.7140e-01],
                           [-1.4744e+00, -3.4327e-01]])
live = context_vecs[0].detach()
x_l1 = inputs.abs().sum(dim=1)
bound = x_l1.max().item() / d_in**0.5
w_max = max(h.W_value.weight.abs().max().item() for h in mha.heads)
print("live == klaim repo (atol 1e-4)?   :", torch.allclose(live, repo_claim, atol=1e-4))
print(f"max |live - klaim repo|           : {(live - repo_claim).abs().max().item():.4f}")
print(f"init nn.Linear: |w| <= 1/sqrt(d_in) = {1 / d_in**0.5:.4f}; max |W_value| run ini = {w_max:.4f}")
print(f"max_t ||x_t||_1 = {x_l1.max().item():.2f} -> batas |z| untuk SEED APA PUN = {bound:.4f}")
print(f"|z| terbesar: live {live.abs().max().item():.4f} | klaim repo {repo_claim.abs().max().item():.4f}")


live == klaim repo (atol 1e-4)?   : False
max |live - klaim repo|           : 0.8320
init nn.Linear: |w| <= 1/sqrt(d_in) = 0.5774; max |W_value| run ini = 0.4590
max_t ||x_t||_1 = 2.08 -> batas |z| untuk SEED APA PUN = 1.2009
|z| terbesar: live 0.7774 | klaim repo 1.4744


**Errata #2 -- output tercetak solusi Exercise 3.2 di repo tidak dapat dihasilkan oleh kode solusinya sendiri.** Ini bukan masalah RNG atau backend, dengan dua bukti.

1. **Bukti untuk seed mana pun.** Dengan dropout 0, setiap output head adalah kombinasi konveks value, sehingga $|z_t| \le \max_s |v_s|$. Dengan `d_out=1`, setiap value adalah $v_s = x_s \cdot w$. Init default `nn.Linear` menjamin $|w_i| \le 1/\sqrt{3} = 0.5774$; run ini mengamati maksimum 0.4590. Maka $|z_t| \le \max_s \|x_s\|_1 / \sqrt{3} = $ 1.2009 untuk *seed mana pun*. Klaim repo memuat 1.4744 di baris terakhir, di atas batas itu. Kode solusi tidak mungkin menghasilkannya.
2. **Bukti di mesin ini.** RNG dan backend di sini mereproduksi output tersimpan repo untuk kelas yang sama (`MultiHeadAttentionWrapper`, seed 123, `d_out=2`, buku hlm. 85) digit-per-digit. Dengan RNG dan backend yang sama, run `d_out=1` memberi 0.7774 sebagai nilai mutlak terbesar.

Jadi kode solusinya benar. Yang salah adalah blok output tercetaknya, kemungkinan besar sisa dari versi kode lain. Penyebab persisnya tidak dapat dipastikan dari repo. Klasifikasinya: errata output tercetak di companion repo, bukan bug kode buku.

### Exercise 3.3 -- Initializing GPT-2 size attention modules

Teks bab yang diterima builder tidak memuat kotak Exercise 3.3. Soal dan solusinya diambil dari `exercise-solutions.ipynb` repo: buat `MultiHeadAttention` seukuran GPT-2 small, yaitu 12 head, `d_in = d_out = 768`, dan konteks 1,024 token.

In [61]:
context_length = 1024
d_in, d_out = 768, 768
num_heads = 12

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)

Modul terbentuk tanpa output. `head_dim` modul ini adalah 64, jadi pembagi skor di dalamnya adalah $\sqrt{64} = 8$, bukan $\sqrt{768}$. Ini nuansa yang dicatat di UTH 1. Blok kedua solusi menghitung parameternya.

In [62]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(mha)

2360064

2,360,064 parameter, sama dengan angka di solusi repo (`2360064  # (2.36 M)`). Rinciannya: $3 \times 768^2 = 1{,}769{,}472$ untuk Q/K/V tanpa bias, ditambah `out_proj` $768^2 + 768 = 590{,}592$. Buffer mask $1024 \times 1024$ tidak terhitung, karena `count_parameters` hanya menjumlahkan `parameters()`.

Solusi repo menutup dengan klaim bahwa GPT-2 berisi 117M parameter dan sebagian besar tidak berada di attention. Angka totalnya sudah dikoreksi di Errata #1. Klaim proporsinya bisa dihitung dari aritmetika 3.6.2, dengan bias QKV seperti GPT-2 asli:

| Komponen GPT-2 small | Parameter | Porsi |
|---|---|---|
| embedding token + posisi | 39,383,808 | 31.6% |
| attention, 12 blok | 28,348,416 | 22.8% |
| MLP, 12 blok | 56,669,184 | 45.5% |
| LayerNorm | 38,400 | 0.03% |
| **total** | **124,439,808** | 100% |

Attention memang minoritas, sekitar seperlima model. Mayoritas ada di MLP dan embedding, persis seperti klaim repo.

## Summary

- **Empat varian attention** dibangun dari listing repo, verbatim dan dalam urutan buku. Varian pertama adalah dot product plus softmax tanpa bobot. Varian kedua scaled dot-product dengan $W_q, W_k, W_v$ (`SelfAttention_v1`/`v2`). Varian ketiga causal mask $-\infty$ plus dropout (`CausalAttention`). Varian keempat multi-head, dalam bentuk wrapper dan dalam bentuk weight split (`MultiHeadAttention`).
- **Kecocokan dengan buku:** 633 nilai numerik dari 34 output cetak cocok digit-per-digit di torch 2.14.1 (macOS arm64, CPU), termasuk kedua mask dropout seed 123. 43/45 sel repo identik dengan output tersimpan repo. Bedanya hanya pada string versi.
- **UTH 1:** std skor mengikuti $\sqrt{d_k}$ (27.5 vs 27.7 di $d_k$=768). Tanpa skala, bobot max naik 0.46 -> 0.986 dan $\|J\|$ (Jacobian softmax terhadap logit-nya) turun 19x (0.368 -> 0.0197). Dengan skala, $\|J\|$ terhadap logit $s/\sqrt{d_k}$ stabil di 0.380-0.386.
- **UTH 2:** token 1-3 bit-identik setelah token 4-6 diganggu (token 4-6 bergeser s.d. 12.59); Jacobian segitiga atas tepat 0.0, tanpa mask 0/36 entri nol.
- **UTH 3:** mask buku = `2*(u64<0.5)` di macOS arm64 tanpa MKL; MPS memberi mask lain (14/36 nol); saat train rowsum bukan 1 (undian 1: 2.00, 0.00, 2.00, 0.98, 0.77, 1.31), rata-rata 10k 1.000; `no_grad` tetap drop, hanya `eval()` yang identitas.
- **UTH 4:** transplantasi bobot: max selisih 6.0e-08; skala GPT-2 (b=2, T=1024, CPU): wrapper 87 ms vs MHA 52 ms vs SDPA 18 ms; buffer mask wrapper 12x lebih besar.

**Errata dan kalibrasi**

1. *Errata #1 (teks buku, hlm. 90):* GPT-2 terkecil berisi 124,439,808 parameter menurut checkpoint resmi dan aritmetika arsitektur, bukan 117 juta. Solusi Exercise 3.3 di repo mengulang angka 117M.
2. *Errata #2 (output tercetak di repo):* output solusi Exercise 3.2 melanggar batas $|z| \le$ 1.2009, yang berlaku untuk seed apa pun, sehingga tidak dapat berasal dari kode solusinya.
3. *Bukan errata (versi):* `dim=1` vs `dim=-1` dan `keys.shape[1]` vs `keys.shape[-1]` antara buku dan repo terbukti setara untuk tensor 2D.
4. *Bukan errata (RNG/backend):* mask dropout dari MPS berbeda dari mask buku. Pada konfigurasi CPU yang diuji, mask buku identik dengan undian uniform float64. Build lain bisa berbeda tanpa ada bug kode.
5. *Cacat cetak dan anotasi:*
   - halaman 80 memuat `0 .0000` dan tanda `)` yang hilang;
   - anotasi Listing 3.3 menyebut `SelfAttention_v1`, padahal kelasnya dibangun di atas `nn.Linear` milik `v2`;
   - caption figure 3.25 menulis `d_out=4` untuk lebar *gabungan*, sedangkan argumennya `d_out=2` per head.

## What comes next

Bab 4 memasang `MultiHeadAttention` dari bab ini ke dalam transformer block, bersama LayerNorm, MLP dengan GELU, dan shortcut connection. Block-block itu ditumpuk menjadi GPT seukuran `GPT_CONFIG_124M`, dan jumlah parameternya dihitung lagi. Hitungan di sana bisa dicocokkan dengan aritmetika 3.6.2 di atas. Dua knob yang menentukan selisihnya adalah weight tying output head dan `qkv_bias`.